# Day 1 Embedding Comparison

Load synthetic DMRC data, embed with three models, and visualize with UMAP.

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import umap
from sentence_transformers import SentenceTransformer

In [ ]:
DATA_FILE = os.path.join("data", "dmrc", "dmrc_Synthetic_Dataset.json")
OUTPUT_DIR = "docs"
IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
MODELS = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
]

In [ ]:
with open(DATA_FILE, "r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
df = df[["id", "source", "category", "text"]]
df.head(10)

In [ ]:
def embed_texts(model_name, texts):
    kwargs = {}
    if "nomic" in model_name:
        kwargs["trust_remote_code"] = True
    model = SentenceTransformer(model_name, **kwargs)
    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=16,
    )
    return embeddings

def build_umap(embeddings, n_neighbors=15, min_dist=0.1, random_state=42):
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="cosine",
        random_state=random_state,
    )
    return reducer.fit_transform(embeddings)

def plot_umap(coords, categories, model_name, filename):
    palette = {
        "contract_clause": "#1f77b4",
        "ncr_description": "#ff7f0e",
        "dpr_narrative": "#2ca02c",
        "unknown": "#7f7f7f",
    }
    plt.figure(figsize=(8, 6))
    for category in sorted(set(categories)):
        idx = [i for i, cat in enumerate(categories) if cat == category]
        plt.scatter(
            coords[idx, 0],
            coords[idx, 1],
            label=category,
            alpha=0.8,
            s=40,
            c=palette.get(category, "#444444"),
        )
    plt.legend()
    plt.title(f"UMAP projection — {model_name}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

In [ ]:
os.makedirs(IMAGE_DIR, exist_ok=True)

summary_rows = []
for model_name in MODELS:
    print(f"Embedding with {model_name}...")
    start = time.time()
    embeddings = embed_texts(model_name, df["text"].tolist())
    latency = ((time.time() - start) / len(df)) * 1000
    dim = embeddings.shape[1]
    device = "GPU" if torch.cuda.is_available() else "CPU"
    summary_rows.append({
        "model": model_name,
        "dim": dim,
        "latency_ms_per_doc": latency,
        "device": device,
    })
    coords = build_umap(embeddings)
    plot_path = os.path.join(IMAGE_DIR, f"umap_{model_name.replace('/', '_')}.png")
    plot_umap(coords, df["category"].tolist(), model_name, plot_path)
    print(f"Saved {plot_path}")

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
plt.figure(figsize=(12, 4))
for i, model_name in enumerate(MODELS):
    img_path = os.path.join(IMAGE_DIR, f"umap_{model_name.replace('/', '_')}.png")
    if os.path.exists(img_path):
        ax = plt.subplot(1, len(MODELS), i + 1)
        ax.imshow(plt.imread(img_path))
        ax.set_title(model_name)
        ax.axis("off")
plt.tight_layout()